Sel 1: muat data


In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from review_nlp.preprocessing import clean_text, preprocess
from review_nlp.preprocessing.slang_normalizer import load_slang

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "data/raw/com.whatsapp_2026-08-19_2026-11-17.parquet")
used = df[df["exclude_reason"].isna()].copy()
print("Ulasan dipakai:", len(used))

Sel 2: contoh sebelum dan sesudah

In [ ]:
for raw in used["text_raw"].sample(10, random_state=42):
    print("SEBELUM:", raw)
    print("SESUDAH:", preprocess(raw), "\n")

In [ ]:
gaya = used["text_raw"].map(lambda t: any(0x1D400 <= ord(c) <= 0x1D7FF for c in t))
print("Ulasan dengan huruf bergaya:", int(gaya.sum()), f"({gaya.mean():.2%})")
used.loc[gaya, "text_raw"].head(10)

In [ ]:
print(used["lang"].value_counts().head(8))
inggris = used[used["lang"] == "en"]
print("\nLabel bahasa 'en' tapi dipakai:", len(inggris), f"({len(inggris) / len(used):.1%})")
inggris[["rating", "text_raw"]].sample(min(15, len(inggris)), random_state=1)

In [ ]:
campuran = used[used["lang"].isin(["tl", "sw", "so", "et", "no", "de"])]
print("Berlabel tl/sw/so/et/no/de tapi dipakai:", len(campuran), f"({len(campuran) / len(used):.1%})")
campuran[["lang", "text_raw"]].sample(20, random_state=3)

### Temuan: deteksi bahasa
- Kolom `lang` hasil `langdetect` **tidak andal untuk teks gaul/pendek**: sekitar 10% ulasan
  berbahasa Indonesia terdeteksi sebagai Tagalog, Swahili, Somali, dll.
- Karena itu filter hanya membuang bahasa yang **jelas asing** (Inggris, Arab, dll.) dengan
  keyakinan tinggi (>= 0,90). Label `tl/sw/so/et/no/de` sengaja tidak dibuang.
- Ulasan berlabel `en` yang lolos hanya 0,7%, mayoritas campuran Inggris-Indonesia (sah).
- Sebagian ulasan memuat bahasa daerah dan gaul; ini dipertahankan dan menjadi keterbatasan.

Sel 3: kandidat kata slang

In [ ]:
slang = load_slang()
counter = Counter(tok for t in used["text_raw"] for tok in clean_text(t).split())

kandidat = [(w, c) for w, c in counter.most_common(500) if w not in slang]
print("Top 150 token yang belum ada di kamus:")
display(pd.DataFrame(kandidat[:150], columns=["token", "frekuensi"]))

pendek = [(w, c) for w, c in kandidat if len(w) <= 5][:100]
print("Token pendek (<= 5 huruf), calon singkatan:")
display(pd.DataFrame(pendek, columns=["token", "frekuensi"]))

Sel 4: tambah kata ke kamus, lalu muat ulang

In [ ]:
load_slang.cache_clear()
print("Jumlah entri kamus:", len(load_slang()))

In [ ]:
cek = ["ndak", "nda", "kagak", "kaga", "ora", "iso", "gak", "gk", "bgt", "banget"]
{w: counter[w] for w in cek}

Sel 5: efek preprocessing

In [ ]:
used["clean"] = used["text_raw"].map(preprocess)

vocab_raw = len({t for s in used["text_raw"] for t in clean_text(s).split()})
vocab_norm = len({t for s in used["clean"] for t in s.split()})
print("Ukuran kosakata sebelum normalisasi slang:", vocab_raw)
print("Ukuran kosakata sesudah normalisasi slang:", vocab_norm)

kosong = used[used["clean"].str.len() == 0]
print("Teks kosong setelah preprocessing:", len(kosong))
kosong[["text_raw"]].head(10)

Sel 6: kata khas negatif vs positif

In [ ]:
neg = Counter(tok for t in used.loc[used["rating"] <= 2, "clean"] for tok in t.split())
pos = Counter(tok for t in used.loc[used["rating"] >= 4, "clean"] for tok in t.split())
vocab = [w for w in set(neg) | set(pos) if neg[w] + pos[w] >= 20]
n_neg, n_pos = sum(neg.values()), sum(pos.values())
skor = {
    w: np.log((neg[w] + 1) / (n_neg + 1)) - np.log((pos[w] + 1) / (n_pos + 1))
    for w in vocab
}
urut = sorted(skor.items(), key=lambda x: x[1])
pd.DataFrame(
    {
        "khas_positif": [w for w, _ in urut[:20]],
        "khas_negatif": [w for w, _ in urut[-20:][::-1]],
    }
)

Sel 7: apakah emoji membawa sentimen

In [ ]:
from review_nlp.preprocessing.cleaner import EMOJI_RE

used["punya_emoji"] = used["text_raw"].map(lambda t: bool(EMOJI_RE.search(t)))
print("Proporsi ulasan beremoji:", round(used["punya_emoji"].mean(), 3))
pd.crosstab(used["punya_emoji"], used["rating"], normalize="index").round(3)

### Temuan: preprocessing
- Normalisasi slang hanya mengurangi kosakata dari 7.226 menjadi 7.170 (0,8%).
  Kamus kecil cukup; TF-IDF dengan bigram sudah menangkap variasi yang sering muncul.
- 6 ulasan (0,1%) kosong setelah dibersihkan dan dibuang saat pelatihan.
- Huruf bergaya Unicode (mis. tulisan tebal) hanya 0,16% dan kini dinormalisasi.
- Ulasan beremoji 7,4%; sedikit condong positif tetapi masih dalam batas ketidakpastian
  sampel, sehingga emoji tetap dibuang. Calon perbaikan: petakan emoji menjadi kata sentimen.